In [1]:
import pandas as pd

In [2]:
size = 24
dataset = 'kddcup'
df = pd.read_csv(f'/root/EdgeCluster/results/{dataset}/tabular/{size}/final_clustering.csv')


In [3]:
df

,duration,src_bytes,dst_bytes,wrong_fragment,urgent,hot,num_failed_logins,num_compromised,root_shell,su_attempted,...,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,cluster,is_included,target,segment,stream,ids
0,0.0,2.610418e-07,0.001057,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,11,True,11,0,-1,1
1,0.0,3.446905e-07,0.000094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,11,True,11,0,-1,2
2,0.0,3.389216e-07,0.000259,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,11,True,11,0,-1,3
3,0.0,3.158461e-07,0.000259,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,11,True,11,0,-1,4
4,0.0,3.129617e-07,0.000394,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,11,True,11,0,-1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494016,0.0,4.470881e-07,0.000365,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.01,0.0,0.0,-1,False,11,49,-1,494017
494017,0.0,4.067060e-07,0.000443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.01,0.0,0.0,-1,False,11,49,-1,494018
494018,0.0,2.927706e-07,0.000233,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.06,0.01,0.0,0.0,-1,False,11,49,-1,494019
494019,0.0,4.196859e-07,0.000233,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.04,0.01,0.0,0.0,-1,False,11,49,-1,494020


In [4]:

cluster = df['cluster']
timestamps = df['ids']
df = df[df.columns[0:-6]]
print(df)

        duration     src_bytes  dst_bytes  wrong_fragment  urgent  hot  \
0            0.0  2.610418e-07   0.001057             0.0     0.0  0.0   
1            0.0  3.446905e-07   0.000094             0.0     0.0  0.0   
2            0.0  3.389216e-07   0.000259             0.0     0.0  0.0   
3            0.0  3.158461e-07   0.000259             0.0     0.0  0.0   
4            0.0  3.129617e-07   0.000394             0.0     0.0  0.0   
...          ...           ...        ...             ...     ...  ...   
494016       0.0  4.470881e-07   0.000365             0.0     0.0  0.0   
494017       0.0  4.067060e-07   0.000443             0.0     0.0  0.0   
494018       0.0  2.927706e-07   0.000233             0.0     0.0  0.0   
494019       0.0  4.196859e-07   0.000233             0.0     0.0  0.0   
494020       0.0  3.158461e-07   0.000239             0.0     0.0  0.0   

        num_failed_logins  num_compromised  root_shell  su_attempted  ...  \
0                     0.0         

In [5]:
from scipy.spatial import distance
from scipy.stats import median_abs_deviation
import numpy as np

def temp_centroid_coherence(X):
    A,B = X[1:,:],X[:-1,:]   
    d = A-B
    d = np.linalg.norm(d, axis=1)
    if len(d):
        return mad(d,False)
    else:
        return 0

def dist2oc(x,M,l):
    d = distance.cdist(M,x)
    clus = np.unique(l)
    meandclus = np.inf * np.ones(len(clus))
    for i,c in enumerate(clus):
        meandclus[i] = np.mean(d[l==c])
    return np.min(meandclus)

def find_knearest(x, v, k, option='nearest'):
    x=x.flatten()
    if option=='random':
        i = np.random.permutation(len(x))
    else:
        i = np.argsort((np.abs(x - v)))
    ind = i[:k]
    return x[ind]

def mad(x, x_is_int=True):
    madx = median_abs_deviation(x, scale = "normal")
    if x_is_int:
        madx = 1 if madx<1 else madx 
    else:
        madx = 1 if madx==0 else madx 
    nmads = np.abs(x-np.median(x))/madx
    outs = len(nmads[nmads>3])
    return outs

def tempsil(t,x,l,s=200,kn=200,c=1):
    # Description: implementation of the Temporal Silhouette index for the
    # internal validation of streaming clustering, FIV, Jun 2022
    #
    # INPUTS
    # t: 1D-array with timestamps
    # x: XD-array with data vectors
    # l: 1D-array with labels
    # s: window-size of the simple-moving-average (SMA)
    # kn: number-of-neighbors of other clusters for calculating beta
    # c: sigma parameter to weight the penalization over contextual outliers [0...1]
    #
    # OUTPUTS
    # k: 1D-array with cluster-labels
    # ts2: 1D-array with quadratic cluster temporal silhouettes
    # TS: global Temporal Silhuette
 
    k = np.unique(l)
    ts = np.zeros(len(k))
    for i,label in enumerate(k):
        tl0 = t[l==label]
        tl1 = np.roll(tl0,-1)
        dtl = (tl1-tl0)[:-1]
        xk = x[l==label,:]
        IAD = 1 / (1 + c * mad(dtl,True)/xk.shape[0])
        wj = np.argwhere(l==label)
        wnt = np.argwhere(l!=label)
        SMA = np.zeros(xk.shape)
        tst = np.zeros(xk.shape)
        for a in range(xk.shape[1]):
            SMA[:,a] = pd.Series(xk[:,a]).rolling(s,min_periods=1, center=True).mean().to_numpy()
        a = np.zeros(len(SMA))
        b = np.zeros(len(SMA))
        for j in range(len(SMA)-1):
            a[j] =  distance.euclidean(xk[j],SMA[j])
            tst[j] = 0
            if len(wnt)>0:
                m = find_knearest(wnt,wj[j],kn,option='nearest')
                b[j] = dist2oc([xk[j]],x[m,:],l[m])
                tst[j] = (b[j] - a[j])/np.max([a[j],b[j]])
        TCD = temp_centroid_coherence(SMA)/xk.shape[0]
        ts[i] = (1 + np.mean(tst)) * IAD * (1 - TCD) - 1
    _,card = np.unique(l,return_counts=True)
    ts2 = np.power(ts,2) * np.sign(ts)
    TS2 = np.sum(card*ts2)/np.sum(card)
    TS = np.sqrt(np.abs(TS2)) * np.sign(TS2)
    return k,ts2,TS



Calculating Temporal Silhouette index

In [ ]:
tempsil(timestamps, df.values, cluster)